In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from typing import List, Dict

1. Data Loading and Cleaning

In [ ]:
df = pd.read_csv('us_tornado_dataset_1950_2021.csv')


print(f"Original rows: {len(df)}")
df = df[df['mag'] >= 0]
print(f"Cleaned rows: {len(df)}")

# Select Features and Target
feature_cols = ['wid', 'len', 'slat', 'slon', 'mo']
X = df[feature_cols].values
y = df['mag'].values

2. Pre-processing

In [ ]:
# Split 80/20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

3. Define Model Architecture

In [ ]:
model = Sequential([
     Dense(64, activation='relu', input_shape=(5,)),
    Dense(6, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

4. Train

In [ ]:
print("Starting training...")
# batch_size=32 means it updates weights after every 32 rows
# verbose=1 shows a progress bar
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

5. Evaluate

In [ ]:
print("\nGenerating Report...")
# Predict returns probabilities for all 6 classes
y_pred_probs = model.predict(X_test)

y_pred = np.argmax(y_pred_probs, axis=1)

print("\n--- Tornado Magnitude Classification Report (TensorFlow) ---\n")
print(classification_report(y_test, y_pred, labels=[0,1,2,3,4,5], zero_division=0))

6. Test

In [ ]:
classes = ('EF0 (Light)', 'EF1 (Moderate)', 'EF2 (Significant)', 
           'EF3 (Severe)', 'EF4 (Devastating)', 'EF5 (Incredible)')

def predict_tornado_damage_tf(tornadoes: List[Dict], model, scaler):

    input_df = pd.DataFrame(tornadoes)
    

    input_df = input_df[feature_cols]
    

    scaled_data = scaler.transform(input_df.values)
    
    probs = model.predict(scaled_data, verbose=0)
    predictions = np.argmax(probs, axis=1)
    
    return [classes[p] for p in predictions]

# --- TEST CASES ---
severe_case = {
    "wid": 2000,
    "len": 25.0,
    "slat": 35.0,
    "slon": -97.0,
    "mo": 5
}

weak_case = {
    "wid": 30,
    "len": 0.2,
    "slat": 41.0,
    "slon": -87.0,
    "mo": 9
}

print("\n--- Live Testing Results (TensorFlow) ---")
results = predict_tornado_damage_tf([severe_case, weak_case], model, scaler)

for i, res in enumerate(results):
    print(f"Test Case #{i+1}: Predicted {res}")